# 00 - Colab GPU Pipeline (Full Fidelity)

Run this on **Google Colab with a T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).

It reproduces the GPU-heavy stages that a CPU-only laptop cannot finish:
1. BERT `[CLS]` + Sentence-Transformer embeddings of the bank rows
2. DistilBERT sentiment (not VADER) + severity + LDA on the CFPB sample
3. Probabilistic fusion
4. Configurations A / B / C with 10-fold CV (all 5 classifiers)
5. SHAP on the best Hybrid model

## Before you start
Upload these to the Colab session (or mount Google Drive):
- `data/bank_churn.csv`
- `data/maven_bank_churn.csv`
- `data/cfpb_complaints.csv`  (8.3 GB - or upload the pre-sampled `cfpb_sample.parquet` instead, ~50 MB, much faster)
- the whole `code/` folder

**Tip:** uploading the 50 MB `data/processed/cfpb_sample.parquet` (produced on your laptop) is far faster than uploading the 8.3 GB raw CSV. This notebook uses it if present.

In [ ]:
import sys
import os

# 1. Uninstall and Install compatible versions
!pip uninstall -y gensim
# Using more recent compatible versions for the current Colab environment
!pip -q install "numpy>=1.26.0" "transformers>=4.44.0" "peft>=0.12.0" "scikit-learn>=1.5.0" sentence-transformers xgboost imbalanced-learn shap

print('Installation complete. numpy>=1.26 is already satisfied by Colab (2.0.2), so NO restart is needed.')
print('You can now Run All, or continue with the cells below.')

Installation complete. numpy>=1.26 is already satisfied by Colab (2.0.2), so NO restart is needed.
You can now Run All, or continue with the cells below.


In [ ]:
# 1b. Verification cell
import numpy, sklearn, torch, xgboost, shap, sentence_transformers, imblearn

print(f'numpy: {numpy.__version__}')
print(f'sklearn: {sklearn.__version__}')
print(f'xgboost: {xgboost.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

print('\nEnvironment verified. You can proceed.')

numpy: 2.0.2
sklearn: 1.6.1
xgboost: 3.3.0
GPU available: True
Device: Tesla T4

Environment verified. You can proceed.


In [ ]:
from google.colab import drive
import sys, os

# 2. Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

# Set this to where you put the thesis folder in Drive
THESIS = '/content/drive/MyDrive/thesis'

if not os.path.isdir(THESIS):
    print(f"ERROR: THESIS directory not found at {THESIS}")
    print("Please check your Drive folder name.")
else:
    # Add the folder containing the 'thesis' package to path
    parent_dir = os.path.dirname(THESIS)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)
    if THESIS not in sys.path:
        sys.path.insert(0, THESIS)

    os.chdir(THESIS)
    print(f'Working directory set to: {os.getcwd()}')

### **Environment Ready**

Google Drive is mounted and dependencies are settled. The notebook will now proceed with embedding generation and model training using the files in your `thesis/code` and `thesis/data` directories.

In [ ]:
# Path verification successful. Variable 'THESIS' is active.
print(f'Active project path: {THESIS}')

Active project path: /content/drive/MyDrive/thesis


In [ ]:
# 3. Bank embeddings - BERT [CLS] (768-d) and Sentence-Transformer (384-d)
import sys, os

# Re-define path after restart
THESIS = "/content/drive/MyDrive/thesis"
parent_dir = os.path.dirname(THESIS)

# Ensure the parent directory is in sys.path so "thesis.code" works as a package
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
if THESIS not in sys.path:
    sys.path.insert(0, THESIS)

# Import modules using the full package path to avoid built-in "code" collision
from thesis.code import config, data_loader, preprocess, tabular_to_text, embeddings
import torch

# Ensure data is loaded and cleaned
bank = preprocess.clean_bank_churn(data_loader.load_bank_churn())
texts = tabular_to_text.render(bank, template="declarative")

print("Generating Sentence-Transformer embeddings...")
emb_st = embeddings.embed_sentence_transformer(texts, batch_size=128, device="cuda")
embeddings.cache_embeddings(emb_st, "bank_emb_st")

print("Generating BERT [CLS] embeddings...")
emb_bert = embeddings.embed_bert_cls(texts, batch_size=64, device="cuda")
embeddings.cache_embeddings(emb_bert, "bank_emb_bert")

print("Done! ST shape:", emb_st.shape, "| BERT shape:", emb_bert.shape)

In [ ]:
# 4. CFPB complaint features with the REAL DistilBERT sentiment model (GPU)
import pandas as pd
# Use thesis.code namespace
from thesis.code import complaint_features, fusion, config, data_loader, preprocess

# Prefer the pre-sampled parquet (fast). Otherwise build it from the raw CSV.
sample_path = config.PROCESSED_DIR / "cfpb_sample.parquet"
if sample_path.exists():
    cfpb = pd.read_parquet(sample_path)
else:
    cfpb = data_loader.load_cfpb_complaints()
    cfpb["narrative"] = cfpb["narrative"].apply(preprocess.clean_narrative)
    cfpb = data_loader.stratified_sample(cfpb, by="product", n=config.CFPB_SAMPLE_SIZE)

cfpb = complaint_features.enrich(cfpb)   # DistilBERT sentiment on GPU + severity + LDA
cfpb.to_parquet(config.PROCESSED_DIR / "cfpb_enriched.parquet", index=False)

ftable = fusion.aggregate_complaint_features(cfpb)
ftable.to_csv(config.PROCESSED_DIR / "fusion_table.csv", index=False)
merged = fusion.join_to_bank(bank, ftable)
merged.to_parquet(config.PROCESSED_DIR / "bank_merged.parquet", index=False)
print("Fusion keys:", len(ftable), " merged:", merged.shape)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Fusion keys: 20  merged: (10000, 19)


In [ ]:
# 5. Configurations A / B / C with 10-fold CV, all classifiers
import numpy as np, pandas as pd
import sys, os
import joblib
from pathlib import Path

# Ensure path persistence
THESIS = "/content/drive/MyDrive/thesis"
parent_dir = os.path.dirname(THESIS)
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)

from thesis.code import classifiers, evaluate, config, preprocess, fusion, embeddings

# Re-load data if memory was cleared by restart
if "merged" not in locals():
    print("Re-loading merged data from disk...")
    merged = pd.read_parquet(config.PROCESSED_DIR / "bank_merged.parquet")

if "emb_bert" not in locals():
    print("Searching for BERT embeddings in cache...")
    # Files found in previous error trace showed .npy extension
    emb_path = config.PROCESSED_DIR / "bank_emb_bert.npy"

    if emb_path.exists():
        print(f"Loading: {emb_path}")
        emb_bert = np.load(emb_path)
    else:
        raise FileNotFoundError(f"BERT embeddings not found at {emb_path}. Ensure Cell 3 ran successfully.")

y = merged[config.TARGET_COL].astype(int).values
drop = [config.TARGET_COL, "product_bucket", "region"]
X_trad, names_trad = preprocess.encode_traditional(merged.drop(columns=[c for c in drop if c in merged.columns]))
for c in fusion.get_complaint_feature_columns():
    if c not in X_trad.columns and c in merged.columns:
        X_trad[c] = merged[c].astype(float).values

X_A = X_trad.values
X_B = emb_bert.astype(float)
X_C = np.concatenate([X_A, X_B], axis=1)

results = []
for cfg, X in [("A", X_A), ("B", X_B), ("C", X_C)]:
    for clf in classifiers.CLASSIFIERS:
        cv = evaluate.cross_validate_metrics(classifiers.make_pipeline(clf), X, y)
        results.append({"config": cfg, "clf": clf, "f1": cv["f1"].mean(), "roc_auc": cv["roc_auc"].mean(), "accuracy": cv["accuracy"].mean(), "mcc": cv["mcc"].mean()})
        print(cfg, clf, "F1=%.3f AUC=%.3f" % (cv["f1"].mean(), cv["roc_auc"].mean()))

pd.DataFrame(results).to_csv(config.PROCESSED_DIR / "colab_all_configs.csv", index=False)

A lr F1=0.562 AUC=0.833
A rf F1=0.609 AUC=0.858
A xgb F1=0.605 AUC=0.855
A svm F1=0.435 AUC=0.760
A ann F1=0.591 AUC=0.859
B lr F1=0.567 AUC=0.832
B rf F1=0.504 AUC=0.795
B xgb F1=0.529 AUC=0.807


In [3]:
import joblib
import sys, os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from google.colab import drive # Added this import to allow auto-mounting

# 1. Debugging: Check Drive content & Auto-Mount
print("Checking Drive contents...")
if not os.path.exists('/content/drive/MyDrive'):
    print("Drive not mounted. Mounting now...")
    drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/drive/MyDrive'):
    if not os.path.exists('/content/drive/MyDrive/thesis'):
        print("CRITICAL WARNING: The 'thesis' folder was not found in your Google Drive.")
        print("Please make sure you have uploaded the 'thesis' folder directly to the root of your MyDrive.")
    else:
        print("Drive is mounted and 'thesis' folder was found.")

# Ensure path persistence
THESIS = "/content/drive/MyDrive/thesis"
parent_dir = os.path.dirname(THESIS)

if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Force clear cache for 'thesis' to prevent ModuleNotFoundError if it failed before
for mod in list(sys.modules):
    if mod == 'thesis' or mod.startswith('thesis.'):
        del sys.modules[mod]

try:
    from thesis.code import shap_explain, classifiers, config, preprocess, embeddings, fusion
    print("Imports successful!")
except ModuleNotFoundError as e:
    print(f"Error: {e}")
    print(f"Target path: {THESIS}")
    print(f"Does it exist? {os.path.isdir(THESIS)}")
    raise

# Re-load data if memory was cleared by restart
if 'merged' not in locals():
    merged = pd.read_parquet(config.PROCESSED_DIR / 'bank_merged.parquet')

if 'emb_bert' not in locals():
    emb_path = config.PROCESSED_DIR / 'bank_emb_bert.npy'
    if emb_path.exists():
        emb_bert = np.load(emb_path)
    else:
        raise FileNotFoundError(f'Could not find bank_emb_bert.npy at {emb_path}')

if 'X_B' not in locals():
    X_B = emb_bert.astype(float)

if 'names_trad' not in locals():
    drop = [config.TARGET_COL, 'product_bucket', 'region']
    X_trad, names_trad = preprocess.encode_traditional(merged.drop(columns=[c for c in drop if c in merged.columns]))
    for c in fusion.get_complaint_feature_columns():
        if c not in X_trad.columns and c in merged.columns:
            X_trad[c] = merged[c].astype(float).values

if 'X_C' not in locals():
    X_A = X_trad.values
    X_C = np.concatenate([X_A, X_B], axis=1)

y = merged[config.TARGET_COL].astype(int).values
names_C = list(names_trad) + [f'bert_{i}' for i in range(X_B.shape[1])]
pipe = classifiers.make_pipeline('xgb')
Xtr, Xte, ytr, yte = train_test_split(X_C, y, test_size=0.15, stratify=y, random_state=42)
pipe.fit(Xtr, ytr)

# Robustly prepare data for SHAP by skipping samplers (like SMOTE)
X_pre = Xte
for name, step in pipe.steps[:-1]:
    if hasattr(step, 'transform'):
        X_pre = step.transform(X_pre)

sv = shap_explain.explain_tree(pipe.named_steps['clf'], X_pre, names_C)

shap_explain.top_features_table(sv, top_k=15).to_csv(config.PROCESSED_DIR / 'shap_top_hybrid.csv', index=False)
shap_explain.save_summary_plot(sv, config.FIGURES_DIR / 'fig_4_6_shap_summary.png')
print('SHAP done. Results saved to Drive.')

Checking Drive contents...
Drive is mounted and 'thesis' folder was found.
Imports successful!
SHAP done. Results saved to Drive.


## After the run

Download these back into your local `thesis/` folder and commit them:
- `data/processed/colab_all_configs.csv`  -> update Tables 4.4-4.6
- `data/processed/shap_top_hybrid.csv`     -> update Table 4.8
- `figures/fig_4_6_shap_summary.png`       -> real SHAP figure

Then re-run `python scripts/make_figures.py` locally to refresh the result figures from the new CSVs.